Imports & Setup

In [ ]:
import os
import glob
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageDraw

# Configure plotting
plt.style.use("ggplot")
sns.set_theme(context="notebook")

# Adjust to your dataset root
DATASET_ROOT = Path(r"C:\Users\Cole.Flashman\source\repos\AutoFactoryScope\project\data")  # <<< UPDATE THIS PER USER

TRAIN_IMG = DATASET_ROOT / "train/images"
VAL_IMG   = DATASET_ROOT / "val/images"
TEST_IMG  = DATASET_ROOT / "test/images"

TRAIN_LBL = DATASET_ROOT / "train/labels"
VAL_LBL   = DATASET_ROOT / "val/labels"
TEST_LBL  = DATASET_ROOT / "test/labels"

print("Dataset root:", DATASET_ROOT)

Validate Folder Structure

In [ ]:
def check_dir(path: Path):
    print(f"{path}  --> {'OK' if path.exists() else '❌ MISSING'}")

check_dir(TRAIN_IMG)
check_dir(TRAIN_LBL)
check_dir(VAL_IMG)
check_dir(VAL_LBL)
check_dir(TEST_IMG)
check_dir(TEST_LBL)

List Image & Label Counts

In [ ]:
def count_files(img_dir, lbl_dir):
    img_count = len(glob.glob(str(img_dir / "*.*")))
    lbl_count = len(glob.glob(str(lbl_dir / "*.txt")))
    return img_count, lbl_count

splits = ["train", "val", "test"]
counts = {
    "split": [],
    "images": [],
    "labels": [],
}

for name, img_dir, lbl_dir in [
    ("train", TRAIN_IMG, TRAIN_LBL),
    ("val", VAL_IMG, VAL_LBL),
    ("test", TEST_IMG, TEST_LBL)
]:
    img_c, lbl_c = count_files(img_dir, lbl_dir)
    counts["split"].append(name)
    counts["images"].append(img_c)
    counts["labels"].append(lbl_c)

df_counts = pd.DataFrame(counts)
df_counts

Plot Image/Label Balance

In [ ]:
df_counts.set_index("split").plot(kind="bar", figsize=(7,5))
plt.title("Dataset Split Balance (Images vs Labels)")
plt.ylabel("Count")
plt.show()

Parse Labels (YOLO Format)

In [14]:
def load_yolo_label(label_path: Path):
    """
    YOLO: class x_center y_center width height (all normalized)
    """
    boxes = []
    if not label_path.exists():
        return boxes
    
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                cls, x, y, w, h = parts
                boxes.append({
                    "class": int(cls),
                    "x": float(x),
                    "y": float(y),
                    "w": float(w),
                    "h": float(h),
                    "label_path": str(label_path)
                })
    return boxes

Load All Labels Into a DataFrame

In [ ]:
def collect_labels(img_dir: Path, lbl_dir: Path):
    all_labels = []
    for img_path in sorted(img_dir.glob("*.*")):
        if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
            continue
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        boxes = load_yolo_label(lbl_path)
        for b in boxes:
            b["image"] = img_path.name
            all_labels.append(b)
    return pd.DataFrame(all_labels)

df_train_labels = collect_labels(TRAIN_IMG, TRAIN_LBL)
df_val_labels = collect_labels(VAL_IMG, VAL_LBL)
df_test_labels = collect_labels(TEST_IMG, TEST_LBL)

df_all = pd.concat([
    df_train_labels.assign(split="train"),
    df_val_labels.assign(split="val"),
    df_test_labels.assign(split="test"),
]).reset_index(drop=True)

df_all

Basic Label Statistics

In [ ]:
print("Total labels:", len(df_all))
print(df_all.groupby("split").size())
print(df_all["class"].value_counts())

Plot Class Distribution

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x="class", data=df_all)
plt.title("Class Distribution")
plt.xlabel("Class ID")
plt.ylabel("Count")
plt.show()

Plot Bounding Box Size Distribution

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(data=df_all, x="w", y="h", hue="split", alpha=0.6)
plt.title("Bounding Box (Width vs Height) — Normalized")
plt.xlabel("Width")
plt.ylabel("Height")
plt.show()

Visualize Random Annotated Images

In [ ]:
def draw_boxes(img_path: Path, lbl_path: Path):
    img = Image.open(img_path).convert("RGB")
    w, h = img.size
    draw = ImageDraw.Draw(img)

    boxes = load_yolo_label(lbl_path)
    for b in boxes:
        # convert normalized YOLO → absolute pixel coords
        x_c = b["x"] * w
        y_c = b["y"] * h
        bw  = b["w"] * w
        bh  = b["h"] * h

        x1 = x_c - bw/2
        y1 = y_c - bh/2
        x2 = x_c + bw/2
        y2 = y_c + bh/2

        draw.rectangle([x1, y1, x2, y2], outline="red", width=3)

    return img

def show_random_samples(split="train", n=4):
    if split == "train":
        img_dir, lbl_dir = TRAIN_IMG, TRAIN_LBL
    elif split == "val":
        img_dir, lbl_dir = VAL_IMG, VAL_LBL
    else:
        img_dir, lbl_dir = TEST_IMG, TEST_LBL

    imgs = sorted(list(img_dir.glob("*.*")))
    samples = random.sample(imgs, min(n, len(imgs)))

    plt.figure(figsize=(10,10))
    for i, img_path in enumerate(samples):
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        img = draw_boxes(img_path, lbl_path)
        plt.subplot(2,2,i+1)
        plt.imshow(img)
        plt.axis("off")
        plt.title(img_path.name)
    plt.show()

show_random_samples("train", 4)

Missing/Empty Label Analysis

In [20]:
def label_status(img_dir, lbl_dir):
    imgs = sorted(img_dir.glob("*.*"))
    missing = []
    empty = []

    for img in imgs:
        lbl = lbl_dir / (img.stem + ".txt")
        if not lbl.exists():
            missing.append(img.name)
        else:
            with open(lbl) as f:
                if len(f.read().strip()) == 0:
                    empty.append(img.name)

    return missing, empty

missing_train, empty_train = label_status(TRAIN_IMG, TRAIN_LBL)
missing_val,   empty_val   = label_status(VAL_IMG, VAL_LBL)
missing_test,  empty_test  = label_status(TEST_IMG, TEST_LBL)

print("Missing labels:", len(missing_train) + len(missing_val) + len(missing_test))
print("Empty labels:", len(empty_train) + len(empty_val) + len(empty_test))

Missing labels: 0
Empty labels: 0


Summary of Findings

In [ ]:
print("===== EDA SUMMARY =====")

print("\nImages per split:")
print(df_counts)

print("\nTotal labels:", len(df_all))
print("Labels per split:")
print(df_all.groupby("split").size())

print("\nMissing label files:")
print(" Train:", len(missing_train))
print(" Val:",   len(missing_val))
print(" Test:",  len(missing_test))

print("\nEmpty label files:")
print(" Train:", len(empty_train))
print(" Val:",   len(empty_val))
print(" Test:",  len(empty_test))